In [1]:
"""
07_boosted_training.ipynb
Дообучение классификатора с русскими разговорными фразами
"""
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import classification_report, f1_score, confusion_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import json
import os

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "cointegrated/rubert-tiny2"

CATEGORIES = ['account', 'delivery', 'general_info', 'order_status', 'other',
              'payment_refund', 'product_info', 'promo_loyalty', 'return_exchange',
              'technical_issue']

CATEGORY_LABELS = {
    'account':          '👤 Аккаунт',
    'delivery':         '🚚 Доставка',
    'general_info':     'ℹ️ Общая информация',
    'order_status':     '📦 Статус заказа',
    'other':            '❓ Прочее',
    'payment_refund':   '💳 Оплата/возврат средств',
    'product_info':     '🛍️ Товар/ассортимент',
    'promo_loyalty':    '🏷️ Промокоды/бонусы',
    'return_exchange':  '🔄 Возврат/обмен товара',
    'technical_issue':  '🔧 Техническая проблема',
}

cat2id = {c: i for i, c in enumerate(CATEGORIES)}
id2cat = {i: c for c, i in cat2id.items()}

print(f"Device: {DEVICE}")
print(f"Категорий: {len(CATEGORIES)}")

Device: cuda
Категорий: 10


In [2]:
"""
Определение Dataset и Model
"""

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True,
            padding="max_length", max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }


class IntentClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)
    
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0, :])
        return self.fc(cls)

print("✅ Классы определены")

✅ Классы определены


In [3]:
"""
Функция обучения и оценки
"""

def train_and_evaluate(model_name, train_df, test_df, epochs=10, batch_size=32, lr=2e-5):
    print(f"\n{'='*60}")
    print(f"🚀 Training: {model_name}")
    print(f"   Train: {len(train_df)}, Test: {len(test_df)}")
    print(f"   Epochs: {epochs}, Batch: {batch_size}, LR: {lr}")
    print(f"   Device: {DEVICE}")
    print(f"{'='*60}")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    train_texts = train_df['text'].tolist()
    train_labels = [cat2id[c] for c in train_df['category']]
    test_texts = test_df['text'].tolist()
    test_labels = [cat2id[c] for c in test_df['category']]
    
    train_loader = DataLoader(
        TextDataset(train_texts, train_labels, tokenizer),
        batch_size=batch_size, shuffle=True
    )
    test_loader = DataLoader(
        TextDataset(test_texts, test_labels, tokenizer),
        batch_size=batch_size
    )
    
    model = IntentClassifier(model_name, len(CATEGORIES)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    train_start = time.time()
    best_f1 = 0
    best_state = None
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for batch in train_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbl = batch["label"].to(DEVICE)
            
            logits = model(ids, mask)
            loss = criterion(logits, lbl)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        
        if (epoch + 1) % 2 == 0 or epoch == epochs - 1:
            model.eval()
            preds = []
            with torch.no_grad():
                for batch in test_loader:
                    logits = model(
                        batch["input_ids"].to(DEVICE),
                        batch["attention_mask"].to(DEVICE)
                    )
                    preds.extend(logits.argmax(dim=1).cpu().tolist())
            
            f1 = f1_score(test_labels, preds, average='macro')
            print(f"  Epoch {epoch+1:2d}/{epochs} | Loss: {avg_loss:.4f} | F1: {f1:.3f}")
            
            if f1 > best_f1:
                best_f1 = f1
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            print(f"  Epoch {epoch+1:2d}/{epochs} | Loss: {avg_loss:.4f}")
    
    train_time = time.time() - train_start
    
    # Финальная оценка
    model.load_state_dict(best_state)
    model.eval()
    
    all_preds, all_labels, all_confs = [], [], []
    
    with torch.no_grad():
        for batch in test_loader:
            logits = model(
                batch["input_ids"].to(DEVICE),
                batch["attention_mask"].to(DEVICE)
            )
            probs = torch.softmax(logits, dim=1)
            confs, preds = probs.max(dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(batch["label"].tolist())
            all_confs.extend(confs.cpu().tolist())
    
    report = classification_report(
        all_labels, all_preds,
        target_names=CATEGORIES, digits=3, output_dict=True
    )
    
    print(f"\n  ⏱️  Время обучения:      {train_time:.1f}s")
    print(f"  🎯 Accuracy:              {report['accuracy']:.3f}")
    print(f"  📊 Macro F1:              {report['macro avg']['f1-score']:.3f}")
    print(f"  📊 Avg Confidence:        {np.mean(all_confs):.3f}")
    print(f"\n{classification_report(all_labels, all_preds, target_names=CATEGORIES, digits=3)}")
    
    # Сохранение
    save_dir = f"../models/{model_name.split('/')[-1]}"
    os.makedirs(save_dir, exist_ok=True)
    torch.save(best_state, f"{save_dir}/best_model.pt")
    print(f"💾 Модель сохранена: {save_dir}/best_model.pt")
    
    return {
        "accuracy": report['accuracy'],
        "macro_f1": report["macro avg"]["f1-score"],
        "avg_confidence": np.mean(all_confs),
        "train_time": train_time,
        "all_preds": all_preds,
        "all_labels": all_labels,
        "report": report,
    }

print("✅ Функция train_and_evaluate определена")

✅ Функция train_and_evaluate определена


In [4]:
"""
200 русских разговорных фраз для дообогащения датасета
"""

russian_boost = [
    # ── order_status (20) ──
    {"text": "где мой заказ", "category": "order_status"},
    {"text": "где посылка", "category": "order_status"},
    {"text": "когда придёт заказ", "category": "order_status"},
    {"text": "когда доставят", "category": "order_status"},
    {"text": "статус заказа", "category": "order_status"},
    {"text": "отследить заказ", "category": "order_status"},
    {"text": "что с моим заказом", "category": "order_status"},
    {"text": "заказ не пришёл", "category": "order_status"},
    {"text": "сколько ждать заказ", "category": "order_status"},
    {"text": "хочу отменить заказ", "category": "order_status"},
    {"text": "отмените мой заказ", "category": "order_status"},
    {"text": "можно отменить заказ?", "category": "order_status"},
    {"text": "где мой товар?", "category": "order_status"},
    {"text": "заказ потерялся", "category": "order_status"},
    {"text": "мой заказ уже отправили?", "category": "order_status"},
    {"text": "дайте трек номер", "category": "order_status"},
    {"text": "как отследить посылку", "category": "order_status"},
    {"text": "уже неделю жду заказ", "category": "order_status"},
    {"text": "долго ждать ещё?", "category": "order_status"},
    {"text": "номер отслеживания", "category": "order_status"},

    # ── delivery (20) ──
    {"text": "как оформить доставку", "category": "delivery"},
    {"text": "сколько стоит доставка", "category": "delivery"},
    {"text": "способы доставки", "category": "delivery"},
    {"text": "бесплатная доставка", "category": "delivery"},
    {"text": "доставка в Москву", "category": "delivery"},
    {"text": "есть самовывоз?", "category": "delivery"},
    {"text": "пункты выдачи", "category": "delivery"},
    {"text": "где забрать заказ", "category": "delivery"},
    {"text": "изменить адрес доставки", "category": "delivery"},
    {"text": "поменять адрес", "category": "delivery"},
    {"text": "доставляете в мой город?", "category": "delivery"},
    {"text": "сроки доставки", "category": "delivery"},
    {"text": "доставка курьером", "category": "delivery"},
    {"text": "стоимость доставки", "category": "delivery"},
    {"text": "бесплатная доставка от какой суммы", "category": "delivery"},
    {"text": "можно доставить сегодня?", "category": "delivery"},
    {"text": "доставка в выходные", "category": "delivery"},
    {"text": "как долго доставка", "category": "delivery"},
    {"text": "доставка почтой", "category": "delivery"},
    {"text": "доставка в регионы", "category": "delivery"},

    # ── payment_refund (20) ──
    {"text": "верните деньги", "category": "payment_refund"},
    {"text": "верните мне деньги", "category": "payment_refund"},
    {"text": "хочу возврат денег", "category": "payment_refund"},
    {"text": "когда вернут деньги", "category": "payment_refund"},
    {"text": "не могу оплатить", "category": "payment_refund"},
    {"text": "ошибка при оплате", "category": "payment_refund"},
    {"text": "как оплатить", "category": "payment_refund"},
    {"text": "способы оплаты", "category": "payment_refund"},
    {"text": "оплата картой", "category": "payment_refund"},
    {"text": "оплата при получении", "category": "payment_refund"},
    {"text": "деньги списали а заказ не оформился", "category": "payment_refund"},
    {"text": "списали два раза", "category": "payment_refund"},
    {"text": "статус возврата денег", "category": "payment_refund"},
    {"text": "не проходит оплата", "category": "payment_refund"},
    {"text": "карта не принимается", "category": "payment_refund"},
    {"text": "дайте чек", "category": "payment_refund"},
    {"text": "возврат средств", "category": "payment_refund"},
    {"text": "можно в рассрочку?", "category": "payment_refund"},
    {"text": "когда придут деньги обратно", "category": "payment_refund"},
    {"text": "пришлите квитанцию", "category": "payment_refund"},

    # ── return_exchange (20) ──
    {"text": "хочу вернуть товар", "category": "return_exchange"},
    {"text": "как вернуть товар", "category": "return_exchange"},
    {"text": "возврат товара", "category": "return_exchange"},
    {"text": "как оформить возврат", "category": "return_exchange"},
    {"text": "можно обменять", "category": "return_exchange"},
    {"text": "обмен товара", "category": "return_exchange"},
    {"text": "товар бракованный", "category": "return_exchange"},
    {"text": "пришёл брак", "category": "return_exchange"},
    {"text": "товар с дефектом", "category": "return_exchange"},
    {"text": "не тот товар прислали", "category": "return_exchange"},
    {"text": "товар не подошёл", "category": "return_exchange"},
    {"text": "хочу обменять на другой размер", "category": "return_exchange"},
    {"text": "сроки возврата", "category": "return_exchange"},
    {"text": "условия возврата", "category": "return_exchange"},
    {"text": "гарантия на товар", "category": "return_exchange"},
    {"text": "гарантийный случай", "category": "return_exchange"},
    {"text": "товар сломался", "category": "return_exchange"},
    {"text": "обменять на другой цвет", "category": "return_exchange"},
    {"text": "вернуть покупку", "category": "return_exchange"},
    {"text": "можно вернуть?", "category": "return_exchange"},

    # ── product_info (20) ──
    {"text": "есть в наличии?", "category": "product_info"},
    {"text": "сколько стоит", "category": "product_info"},
    {"text": "цена товара", "category": "product_info"},
    {"text": "какие размеры есть", "category": "product_info"},
    {"text": "размерная сетка", "category": "product_info"},
    {"text": "из чего сделано", "category": "product_info"},
    {"text": "состав ткани", "category": "product_info"},
    {"text": "характеристики товара", "category": "product_info"},
    {"text": "когда появится в наличии", "category": "product_info"},
    {"text": "есть в другом цвете?", "category": "product_info"},
    {"text": "что лучше выбрать", "category": "product_info"},
    {"text": "подойдёт ли мне", "category": "product_info"},
    {"text": "будет ли скидка на этот товар", "category": "product_info"},
    {"text": "товар в наличии?", "category": "product_info"},
    {"text": "описание товара", "category": "product_info"},
    {"text": "этот товар оригинальный?", "category": "product_info"},
    {"text": "подробнее о товаре", "category": "product_info"},
    {"text": "есть ли этот товар", "category": "product_info"},
    {"text": "нет в наличии когда будет", "category": "product_info"},
    {"text": "отличие моделей", "category": "product_info"},

    # ── account (20) ──
    {"text": "не могу войти", "category": "account"},
    {"text": "забыл пароль", "category": "account"},
    {"text": "как сменить пароль", "category": "account"},
    {"text": "сбросить пароль", "category": "account"},
    {"text": "не приходит код", "category": "account"},
    {"text": "хочу удалить аккаунт", "category": "account"},
    {"text": "как зарегистрироваться", "category": "account"},
    {"text": "регистрация", "category": "account"},
    {"text": "создать аккаунт", "category": "account"},
    {"text": "изменить email", "category": "account"},
    {"text": "поменять телефон", "category": "account"},
    {"text": "войти в личный кабинет", "category": "account"},
    {"text": "проблема со входом", "category": "account"},
    {"text": "аккаунт заблокирован", "category": "account"},
    {"text": "как войти в аккаунт", "category": "account"},
    {"text": "логин и пароль", "category": "account"},
    {"text": "изменить данные профиля", "category": "account"},
    {"text": "удалите мой профиль", "category": "account"},
    {"text": "не могу зарегистрироваться", "category": "account"},
    {"text": "как выйти из аккаунта", "category": "account"},

    # ── promo_loyalty (20) ──
    {"text": "дай промокод", "category": "promo_loyalty"},
    {"text": "промокод", "category": "promo_loyalty"},
    {"text": "есть промокоды?", "category": "promo_loyalty"},
    {"text": "промокод не работает", "category": "promo_loyalty"},
    {"text": "как ввести промокод", "category": "promo_loyalty"},
    {"text": "купон на скидку", "category": "promo_loyalty"},
    {"text": "скидки", "category": "promo_loyalty"},
    {"text": "какие акции", "category": "promo_loyalty"},
    {"text": "есть скидки?", "category": "promo_loyalty"},
    {"text": "распродажа", "category": "promo_loyalty"},
    {"text": "бонусные баллы", "category": "promo_loyalty"},
    {"text": "сколько у меня бонусов", "category": "promo_loyalty"},
    {"text": "программа лояльности", "category": "promo_loyalty"},
    {"text": "как получить скидку", "category": "promo_loyalty"},
    {"text": "промокод не применяется", "category": "promo_loyalty"},
    {"text": "где взять промокод", "category": "promo_loyalty"},
    {"text": "как активировать купон", "category": "promo_loyalty"},
    {"text": "когда будет распродажа", "category": "promo_loyalty"},
    {"text": "чёрная пятница", "category": "promo_loyalty"},
    {"text": "скидка по промокоду", "category": "promo_loyalty"},

    # ── technical_issue (20) ──
    {"text": "сайт не работает", "category": "technical_issue"},
    {"text": "ошибка на сайте", "category": "technical_issue"},
    {"text": "сайт глючит", "category": "technical_issue"},
    {"text": "не грузится страница", "category": "technical_issue"},
    {"text": "приложение вылетает", "category": "technical_issue"},
    {"text": "баг в приложении", "category": "technical_issue"},
    {"text": "не могу добавить в корзину", "category": "technical_issue"},
    {"text": "корзина пустая", "category": "technical_issue"},
    {"text": "ошибка 500", "category": "technical_issue"},
    {"text": "не могу оформить заказ на сайте", "category": "technical_issue"},
    {"text": "кнопка не работает", "category": "technical_issue"},
    {"text": "фото не загружается", "category": "technical_issue"},
    {"text": "белый экран", "category": "technical_issue"},
    {"text": "приложение тормозит", "category": "technical_issue"},
    {"text": "не открывается сайт", "category": "technical_issue"},
    {"text": "ваш сайт лагает", "category": "technical_issue"},
    {"text": "ничего не работает на сайте", "category": "technical_issue"},
    {"text": "висит загрузка", "category": "technical_issue"},
    {"text": "вылетает при оплате", "category": "technical_issue"},
    {"text": "не работает фильтр", "category": "technical_issue"},

    # ── general_info (20) ──
    {"text": "ваш телефон", "category": "general_info"},
    {"text": "номер телефона", "category": "general_info"},
    {"text": "как связаться", "category": "general_info"},
    {"text": "контакты", "category": "general_info"},
    {"text": "режим работы", "category": "general_info"},
    {"text": "во сколько работаете", "category": "general_info"},
    {"text": "график работы", "category": "general_info"},
    {"text": "адрес магазина", "category": "general_info"},
    {"text": "где вы находитесь", "category": "general_info"},
    {"text": "email поддержки", "category": "general_info"},
    {"text": "работаете в выходные?", "category": "general_info"},
    {"text": "ваша почта", "category": "general_info"},
    {"text": "куда обратиться", "category": "general_info"},
    {"text": "телефон горячей линии", "category": "general_info"},
    {"text": "где находится офис", "category": "general_info"},
    {"text": "часы работы поддержки", "category": "general_info"},
    {"text": "реквизиты компании", "category": "general_info"},
    {"text": "юридический адрес", "category": "general_info"},
    {"text": "есть ли офис", "category": "general_info"},
    {"text": "как написать жалобу", "category": "general_info"},

    # ── other (20) ──
    {"text": "оператор", "category": "other"},
    {"text": "позовите менеджера", "category": "other"},
    {"text": "хочу поговорить с оператором", "category": "other"},
    {"text": "соедините с человеком", "category": "other"},
    {"text": "спасибо", "category": "other"},
    {"text": "спасибо за помощь", "category": "other"},
    {"text": "хочу пожаловаться", "category": "other"},
    {"text": "жалоба", "category": "other"},
    {"text": "ужасный сервис", "category": "other"},
    {"text": "отстой", "category": "other"},
    {"text": "помогите", "category": "other"},
    {"text": "мне нужна помощь", "category": "other"},
    {"text": "у меня проблема", "category": "other"},
    {"text": "хочу оставить отзыв", "category": "other"},
    {"text": "привет", "category": "other"},
    {"text": "здравствуйте", "category": "other"},
    {"text": "что вы умеете", "category": "other"},
    {"text": "чем можете помочь", "category": "other"},
    {"text": "вы лучшие", "category": "other"},
    {"text": "не знаю что делать", "category": "other"},
]

boost_df = pd.DataFrame(russian_boost)

print(f"✅ Русских фраз: {len(boost_df)}")
print(f"\nРаспределение:")
print(boost_df['category'].value_counts().sort_index().to_string())

✅ Русских фраз: 200

Распределение:
category
account            20
delivery           20
general_info       20
order_status       20
other              20
payment_refund     20
product_info       20
promo_loyalty      20
return_exchange    20
technical_issue    20


In [5]:
"""
Объединение с основным датасетом и обучение
"""

# Загрузка основного датасета
train_df = pd.read_csv("../data/final/train.csv")
test_df = pd.read_csv("../data/final/test.csv")

print(f"📊 Исходный train: {len(train_df)}")

# Объединение
train_boosted = pd.concat([train_df, boost_df[['text', 'category']]], ignore_index=True)

# Дедупликация
train_boosted['text_lower'] = train_boosted['text'].str.lower().str.strip()
before = len(train_boosted)
train_boosted = train_boosted.drop_duplicates(subset='text_lower').drop(columns='text_lower')

# Перемешивание
train_boosted = train_boosted.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"📊 Boosted train: {len(train_boosted)} (+{len(train_boosted) - len(train_df)})")
print(f"   Дубликатов удалено: {before - len(train_boosted)}")
print(f"\nРаспределение:")
print(train_boosted['category'].value_counts().sort_index().to_string())

# Сохранение
train_boosted.to_csv("../data/final/train_boosted.csv", index=False, encoding="utf-8")

# Обучение
result = train_and_evaluate(
    MODEL_NAME, train_boosted, test_df,
    epochs=10, batch_size=32, lr=2e-5
)

📊 Исходный train: 2322
📊 Boosted train: 2519 (+197)
   Дубликатов удалено: 3

Распределение:
category
account            256
delivery           254
general_info       250
order_status       256
other              256
payment_refund     257
product_info       257
promo_loyalty      233
return_exchange    256
technical_issue    244

🚀 Training: cointegrated/rubert-tiny2
   Train: 2519, Test: 581
   Epochs: 10, Batch: 32, LR: 2e-05
   Device: cuda


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Epoch  1/10 | Loss: 2.1883
  Epoch  2/10 | Loss: 1.5326 | F1: 0.802
  Epoch  3/10 | Loss: 0.9027
  Epoch  4/10 | Loss: 0.5916 | F1: 0.871
  Epoch  5/10 | Loss: 0.4206
  Epoch  6/10 | Loss: 0.3161 | F1: 0.925
  Epoch  7/10 | Loss: 0.2437
  Epoch  8/10 | Loss: 0.1971 | F1: 0.934
  Epoch  9/10 | Loss: 0.1558
  Epoch 10/10 | Loss: 0.1297 | F1: 0.944

  ⏱️  Время обучения:      45.9s
  🎯 Accuracy:              0.943
  📊 Macro F1:              0.944
  📊 Avg Confidence:        0.937

                 precision    recall  f1-score   support

        account      0.983     0.967     0.975        60
       delivery      0.881     0.897     0.889        58
   general_info      0.963     0.912     0.937        57
   order_status      0.965     0.932     0.948        59
          other      0.915     0.915     0.915        59
 payment_refund      0.950     0.950     0.950        60
   product_info      0.875     0.933     0.903        60
  promo_loyalty      1.000     0.981     0.990        53
re

In [6]:
"""
Загрузка обученной модели и функция predict
"""

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = IntentClassifier(MODEL_NAME, len(CATEGORIES)).to(DEVICE)
model.load_state_dict(torch.load("../models/rubert-tiny2/best_model.pt", map_location=DEVICE))
model.eval()

def predict(text):
    encoded = tokenizer(
        text, truncation=True, padding="max_length",
        max_length=128, return_tensors="pt"
    )
    
    with torch.no_grad():
        logits = model(
            encoded["input_ids"].to(DEVICE),
            encoded["attention_mask"].to(DEVICE)
        )
        probs = torch.softmax(logits, dim=1)[0]
    
    top3 = probs.argsort(descending=True)[:3]
    top_cat = CATEGORIES[top3[0]]
    top_conf = probs[top3[0]].item()
    
    print(f"{'─'*50}")
    print(f"📝 «{text}»")
    
    if top_conf >= 0.65:
        print(f"✅ {CATEGORY_LABELS[top_cat]}  ({top_conf:.1%})")
    elif top_conf >= 0.40:
        print(f"⚠️ {CATEGORY_LABELS[top_cat]}  ({top_conf:.1%}) — уточнение")
    else:
        print(f"❌ Не определено ({top_conf:.1%}) — эскалация")
    
    for rank, idx in enumerate(top3):
        cat = CATEGORIES[idx]
        conf = probs[idx].item()
        bar = "█" * int(conf * 30)
        print(f"   {rank+1}. {CATEGORY_LABELS[cat]:30s} {conf:6.1%} {bar}")
    print()

print("✅ Модель загружена, функция predict готова")

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Модель загружена, функция predict готова


In [7]:
"""
Тестирование на проблемных фразах
"""

test_messages = [
    # Проблемные из прошлого теста
    "Как оформить доставку",
    "Здравствуйте, как могу узнать где мой товар?",
    "Дай промокод",
    "Верните мне мои деньги!",
    "Как я могу войти на сайт?",
    
    # Дополнительные
    "хочу вернуть покупку",
    "сколько стоит доставка",
    "промокод не работает",
    "забыл пароль",
    "сайт глючит",
    "где ваш офис",
    "оператор",
    "спасибо",
    "есть в наличии?",
    "когда придёт заказ",
]

for msg in test_messages:
    predict(msg)

──────────────────────────────────────────────────
📝 «Как оформить доставку»
✅ 🚚 Доставка  (91.7%)
   1. 🚚 Доставка                      91.7% ███████████████████████████
   2. 📦 Статус заказа                  7.1% ██
   3. 🏷️ Промокоды/бонусы              0.3% 

──────────────────────────────────────────────────
📝 «Здравствуйте, как могу узнать где мой товар?»
✅ 📦 Статус заказа  (83.4%)
   1. 📦 Статус заказа                 83.4% █████████████████████████
   2. 🚚 Доставка                       6.8% ██
   3. 🛍️ Товар/ассортимент             6.1% █

──────────────────────────────────────────────────
📝 «Дай промокод»
✅ 🏷️ Промокоды/бонусы  (96.9%)
   1. 🏷️ Промокоды/бонусы             96.9% █████████████████████████████
   2. 🔄 Возврат/обмен товара           0.9% 
   3. ℹ️ Общая информация              0.7% 

──────────────────────────────────────────────────
📝 «Верните мне мои деньги!»
✅ 💳 Оплата/возврат средств  (96.6%)
   1. 💳 Оплата/возврат средств        96.6% ██████████████████████

In [25]:
"""
Свободное тестирование — пиши свои сообщения
"""

# Вариант 1: по одному
predict("У вас есть футболки?")

──────────────────────────────────────────────────
📝 «У вас есть футболки?»
✅ ℹ️ Общая информация  (72.6%)
   1. ℹ️ Общая информация             72.6% █████████████████████
   2. 🛍️ Товар/ассортимент            11.2% ███
   3. 🏷️ Промокоды/бонусы              6.9% ██

